### We'll first INSTALL, VERIFY, and DOWNLOAD an Ollama model

In [1]:
!pip install ollama


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!ollama --version

ollama version is 0.32.14


In [3]:
!ollama list

NAME                       ID              SIZE      MODIFIED    
qwen3:0.6b                 7df6b6e09427    522 MB    2 days ago     
nomic-embed-text:latest    0a109f422b47    274 MB    7 weeks ago    
minimax-m3:cloud           8cd948b96f47    -         7 weeks ago    


In [4]:
!ollama pull qwen3:0.6b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 7f4030143c1c: 100% ▕██████████████████▏ 522 MB                         
pulling ae370d884f10: 100% ▕██████████████████▏ 1.7 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling cff3f395ef37: 100% ▕██████████████████▏  120 B                         
pulling b0830f4ff6a0: 100% ▕██████████████████▏  490 B                         
verifying sha256 digest 
writing manifest 
success 


In [5]:
!ollama list

NAME                       ID              SIZE      MODIFIED       
qwen3:0.6b                 7df6b6e09427    522 MB    13 seconds ago    
nomic-embed-text:latest    0a109f422b47    274 MB    7 weeks ago       
minimax-m3:cloud           8cd948b96f47    -         7 weeks ago       


### here, we have downloaded the Ollama model locally, all set Now!

##### We'll use openai-sdk OR Anthropic SDK to use it as build our app, and to save API cost, we'll use Ollama model (we're broke!)

In [6]:
!pip install anthropic

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 11.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


##### on one powershell run: 

`ollama serve` 

##### On another:
`ollama run qwen3:0.6b`

##### Here, we'll define our model initialization. 

In [1]:
import anthropic

client = anthropic.Anthropic(
    base_url="http://localhost:11434",
    api_key="ollama",
    timeout=120.0,
    max_retries=2,
)

##### Here I will show you the simplest execution of a command. 

In [23]:

def generate_claude_answer(input_query: str)-> str:

    message = client.messages.create(
        max_tokens=300,
        system="ONLY ANSWER IN 1000 WORD",
        extra_body={
                "temperature": 0.1,
                "top_p": 0.8,
                "top_k": 20,
                "num_ctx":20000
            },

        messages=[
            {
                "role": "user",
                "content": input_query,
            }
        ],
        model="qwen3:0.6b",
    )

    for block in message.content:
            if block.type == "text":
                print(block.text)
                print("\n")

    return message.content

In [27]:
print(generate_claude_answer("Write me a completely original story. Give me at least 900 words of this story. ") )

[ThinkingBlock(signature=None, thinking="Okay, the user wants me to write a completely original story with at least 900 words. Let me start by brainstorming some ideas. First, I need a unique setting. Maybe a futuristic city? Or a place with some magical element. Let me think about a setting that's both familiar and has a twist. A city with a unique technology or a magical element could work.\n\nThe story needs to be original. Let me create a setting. Maybe a city where time is a resource. That's a good idea. Time as a commodity. Then, introduce a protagonist who has a unique ability. Maybe someone who can manipulate time. Let's say a scientist or a character with a time-traveling ability. \n\nNow, the plot. The protagonist could be someone who's been using time to help others. But there's a problem. Maybe the city's time is over, and the protagonist has to find a way to save it. That adds a conflict. Let me outline the story: start with the city's time being over, introduce the protag

In [28]:
def generate_stream_claude_answer(input_query: str) -> str:

    full_response = []

    with client.messages.stream(
        model="qwen3:0.6b",
        max_tokens=2000,
        system="Answer in approximately 1000 words.",
        extra_body={
            "think": False,
            "temperature": 0.1,
            "top_p": 0.8,
            "top_k": 20,
            "num_ctx": 20000,
        },
        messages=[
            {
                "role": "user",
                "content": input_query,
            }
        ],
    ) as stream:

        for text in stream.text_stream:
            print(text, end="", flush=True)
            full_response.append(text)

    print("\n")

    return "".join(full_response)

In [29]:
print(generate_stream_claude_answer("Write me a completely original story. Give me at least 900 words of this story. "))

**Title: "The Luminous Loom"**

In the heart of the floating world of Luminara, where the sky shimmered like liquid gold and the land pulsed with life, there lived a young woman named Lila. Lila was the daughter of a renowned scientist, Dr. Elara Voss, a pioneer in the field of quantum entanglement. Her father, a man of science, had once been a guardian of the Luminara, but after a mysterious disappearance, Elara had taken her on a quest to find the source of the world’s magic. Now, as Lila navigated the world’s floating islands, she faced a challenge that would test her resolve and her own soul.

Lila’s journey began in the heart of the first island, Virellia, a world of crystalline forests and sentient trees. Each island was a living entity, and Lila had to learn to read their whispers. One such island, Kaelith, was said to be the heart of the world’s magic. But as Lila explored Kaelith, she discovered that the trees were not just alive—they were alive with emotions. One day, while m